# Beyond Audio — Final Experiments, Stage 5
## Reproduce research figures and audit paper-to-code consistency

**Provenance:** reconstructed from the team's saved original test predictions, Stage 3 bootstrap files, Stage 4 sensitivity audit, and the 21-page *Beyond Audio: Music Genre Classification* report (November 2026). Figures are **generated afresh**; they are not copied from the PDF.

**Important:** The Stage 1 retrained CNN Early Fusion and All Four LR models each differed from the original by **one test prediction**. To reproduce figures and values actually reported in the paper, this notebook deliberately uses the **original frozen model-prediction files** rather than the changed retrained versions. It also verifies those files against the Stage 3 reconstruction.

**Outputs:** new `reproducible_stage5_2026` folder; no input files or existing report figures are overwritten. Does not train models, tune settings, download album images, or rerun the 3,000 bootstrap samples (these were reconstructed and verified in Stage 3).

Run each section in order and inspect the output. A comparison that fails should be investigated; do **not** change a source result merely to force a match.

## 1. Connect Google Drive and locate the frozen verification inputs

This checks the dataset, archived predictions, Stage 3 bootstrap results, and Stage 4 robustness files. The new output directory is **not created yet**.

In [ ]:
from pathlib import Path
import os, sys, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, accuracy_score, balanced_accuracy_score, f1_score

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT = Path(os.environ.get("MSDI_PROJECT_DIR", "/content/drive/MyDrive/MSD-I_project"))
EXP = PROJECT / "finetuning_experiments"
CNN_DIR = EXP / "multimodal_artist_disjoint"
MM_DIR = EXP / "oramas_representation_experiment"
ORIGINAL_NN_DIR = EXP / "neural_network_experiments"
STAGE2_DIR = EXP / "reproducible_stage2_2026"
STAGE3_DIR = EXP / "reproducible_stage3_2026"
STAGE4_DIR = EXP / "reproducible_stage4_2026"
ORIGINAL_FIG_DIR = EXP / "final_report_figures"
ORIGINAL_EXAMPLES_DIR = EXP / "phase4_album_examples"
OUTPUT = EXP / "reproducible_stage5_2026"
FIG_DIR = OUTPUT / "figures"

required = {
    "test labels": CNN_DIR / "labels_test.csv",
    "original CNN predictions": CNN_DIR / "final_test_predictions.csv",
    "original representation predictions": MM_DIR / "representation_final_test_predictions.csv",
    "original NN predictions": ORIGINAL_NN_DIR / "neural_network_test_predictions.csv",
    "verified Stage 3 predictions": STAGE3_DIR / "stage3_paired_test_predictions_and_typicality.csv",
    "verified Stage 3 overall transitions": STAGE3_DIR / "stage3_overall_transitions.csv",
    "verified Stage 3 quintile transitions": STAGE3_DIR / "stage3_quintile_transitions.csv",
    "verified Stage 3 bootstrap intervals": STAGE3_DIR / "stage3_quintile_fusion_gain_cluster_bootstrap.csv",
    "verified Stage 4 overall sensitivity": STAGE4_DIR / "stage4_overall_sensitivity.csv",
    "verified Stage 4 quintile sensitivity": STAGE4_DIR / "stage4_quintile_sensitivity.csv",
}
missing = [(name, str(path)) for name, path in required.items() if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(f"{name}: {path}" for name, path in missing))
print("Found all", len(required), "required input files.")
print("Project:", PROJECT)
print("Stage 5 output (not yet created):", OUTPUT)
print("Python:", platform.python_version(), "NumPy:", np.__version__, "pandas:", pd.__version__)

## 2. Build the single verified figure dataset (1,792 test tracks)

Align everything by unique `msd_track_id` and confirm that the original predictions agree with Stage 3. Check both genre and artist IDs; never rely on CSV row ordering.

In [ ]:
GENRES = ["Electronic", "Jazz", "Metal", "Pop", "Rap", "Rock"]
META = ["msd_track_id", "msd_artist_id", "genre", "split"]
dtype_id = {"msd_track_id": str, "msd_artist_id": str}
labels = pd.read_csv(required["test labels"], dtype=dtype_id)
assert len(labels) == 1792 and labels["msd_track_id"].is_unique
assert (labels["split"] == "test").all()
assert sorted(labels["genre"].unique()) == GENRES

model_sources = [
    (required["original CNN predictions"], {
        "CNN Audio LR": "audio_prediction",
        "CNN Visual LR": "visual_prediction",
        "CNN Fixed Late Fusion": "late_fusion_prediction",
        "CNN Adaptive Late Fusion": "adaptive_fusion_prediction",
    }),
    (required["original representation predictions"], {
        "CNN Early Fusion LR": "cnn_early_fusion_prediction",
        "MM Audio LR": "mm_audio_prediction",
        "MM Early Fusion LR": "mm_fusion_prediction",
        "All Four LR": "all_four_prediction",
    }),
    (required["original NN predictions"], {
        "CNN Fusion NN": "cnn_nn_prediction",
        "MM Fusion NN": "mm_nn_prediction",
    }),
]
figure_data = labels[META].copy()
for path, model_columns in model_sources:
    raw = pd.read_csv(path, dtype=dtype_id)
    assert len(raw) == 1792 and raw["msd_track_id"].is_unique, path
    assert set(model_columns.values()).issubset(raw.columns), (path.name, set(model_columns.values()) - set(raw.columns))
    source_metadata = raw[["msd_track_id", "genre"]].merge(
        labels[["msd_track_id", "genre"]], on="msd_track_id", how="outer",
        validate="one_to_one", indicator=True, suffixes=("_source", "_labels")
    )
    assert len(source_metadata) == 1792 and (source_metadata["_merge"] == "both").all(), path
    assert (source_metadata["genre_source"] == source_metadata["genre_labels"]).all(), path
    renamed = raw[["msd_track_id"] + list(model_columns.values())].rename(columns={v:k for k,v in model_columns.items()})
    figure_data = figure_data.merge(renamed, on="msd_track_id", how="left", validate="one_to_one", sort=False)

stage3 = pd.read_csv(required["verified Stage 3 predictions"], dtype=dtype_id)
assert len(stage3) == 1792 and stage3["msd_track_id"].is_unique
stage3_fields = ["msd_track_id", "msd_artist_id", "genre", "typicality_quintile",
                 "source_cnn_audio_prediction", "source_cnn_early_fusion_prediction",
                 "source_mm_mm_audio_prediction", "source_mm_mm_fusion_prediction"]
assert set(stage3_fields).issubset(stage3.columns)
joined = figure_data.merge(stage3[stage3_fields], on="msd_track_id", how="left",
                           suffixes=("", "_stage3"), validate="one_to_one", sort=False)
assert joined.notna().all().all()
for field in ("msd_artist_id", "genre"):
    assert (joined[field] == joined[f"{field}_stage3"]).all(), field
for model, src in {
    "CNN Audio LR": "source_cnn_audio_prediction",
    "CNN Early Fusion LR": "source_cnn_early_fusion_prediction",
    "MM Audio LR": "source_mm_mm_audio_prediction",
    "MM Early Fusion LR": "source_mm_mm_fusion_prediction"
}.items():
    assert (joined[model] == joined[src]).all(), model
for model_columns in (group for _, group in model_sources):
    for model in model_columns:
        assert set(figure_data[model]).issubset(set(GENRES)), model
figure_data["typicality_quintile"] = joined["typicality_quintile"].astype(int)
assert figure_data["typicality_quintile"].value_counts().sort_index().tolist() == [361, 357, 357, 357, 360]
print("Original frozen predictions aligned by track ID: VERIFIED")
print("Matching Stage 3 predictions and quintiles: VERIFIED")
print("Test recordings:", len(figure_data), "| Artists:", figure_data.msd_artist_id.nunique())
print("Models available:", len([m for _, cols in model_sources for m in cols]))
print("Quintile sizes:", figure_data.typicality_quintile.value_counts().sort_index().to_dict())

## 3. Recalculate **Table 2** directly from the original predictions

Check accuracy, balanced accuracy, and macro-F1 for all **10** rows against the November report. Values in the report are rounded to 0.01 percentage points. This audit does not silently correct discrepancies.

In [ ]:
PAPER_TABLE2 = {
    "CNN Audio LR": (65.90, 62.36, 64.07),
    "CNN Visual LR": (47.60, 44.76, 46.34),
    "CNN Early Fusion LR": (69.64, 67.39, 68.68),
    "CNN Fixed Late Fusion": (68.69, 65.02, 67.29),
    "CNN Adaptive Late Fusion": (68.30, 64.97, 67.08),
    "MM Audio LR": (66.52, 63.13, 64.64),
    "MM Early Fusion LR": (70.20, 67.30, 69.10),
    "All Four LR": (69.48, 67.52, 68.69),
    "CNN Fusion NN": (68.58, 67.90, 67.97),
    "MM Fusion NN": (69.03, 68.31, 68.49),
}
y_true = figure_data["genre"].to_numpy()
metric_rows = []
for model, original in PAPER_TABLE2.items():
    pred = figure_data[model].to_numpy()
    actual = np.array([100 * accuracy_score(y_true, pred),
                       100 * balanced_accuracy_score(y_true, pred),
                       100 * f1_score(y_true, pred, labels=GENRES, average="macro", zero_division=0)])
    difference = actual - np.array(original)
    metric_rows.append(dict(model=model, accuracy=actual[0], balanced_accuracy=actual[1],
                            macro_f1=actual[2], max_abs_report_diff_pp=np.max(np.abs(difference))))
metrics = pd.DataFrame(metric_rows)
print("Table 2: computed vs report, percentage points")
print(metrics.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
max_difference = float(metrics["max_abs_report_diff_pp"].max())
print("Maximum absolute difference from rounded report:", f"{max_difference:.6f} pp")
assert max_difference < 0.0051, "Table 2 mismatch: investigate prediction provenance before proceeding"
print("TABLE 2: ALL 10 ROWS MATCH THE REPORT AT TWO DECIMAL PLACES")

## 4. Reconstruct **Figure 1** (row-normalized confusion matrices) and **Table 7** (genre F1)

These use exactly four models named in the report. The original PDF Figure 1 is a **2×2** arrangement of class-row percentages. Save the new figure separately; preserve the original.

In [ ]:
FIGURE1_MODELS = ["CNN Audio LR", "CNN Early Fusion LR", "MM Early Fusion LR", "MM Fusion NN"]
FIGURE1_LABELS = ["CNN Audio LR", "CNN Fusion LR", "MM Fusion LR", "MM Fusion NN"]
PAPER_F1 = {
    "Electronic": [67.86, 72.55, 71.48, 70.97],
    "Jazz": [58.39, 60.78, 61.54, 61.50],
    "Metal": [65.20, 66.20, 69.31, 68.81],
    "Pop": [44.44, 56.21, 55.88, 54.58],
    "Rap": [76.28, 83.23, 81.68, 82.76],
    "Rock": [72.24, 73.07, 74.67, 72.34],
}
figure1_matrices = {}
f1_rows = []
for model, short in zip(FIGURE1_MODELS, FIGURE1_LABELS):
    cm_counts = confusion_matrix(y_true, figure_data[model], labels=GENRES)
    cm_pct = 100 * cm_counts / cm_counts.sum(axis=1, keepdims=True)
    assert np.allclose(cm_pct.sum(axis=1), 100)
    figure1_matrices[short] = (cm_counts, cm_pct)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, figure_data[model], labels=GENRES, zero_division=0)
    for genre, score, n in zip(GENRES, f1, support):
        f1_rows.append(dict(genre=genre, model=short, f1_pct=100*score, support=int(n)))
f1_detail = pd.DataFrame(f1_rows)
f1_table = f1_detail.pivot(index="genre", columns="model", values="f1_pct").loc[GENRES, FIGURE1_LABELS]
reference_f1 = pd.DataFrame.from_dict(PAPER_F1, orient="index", columns=FIGURE1_LABELS).loc[GENRES]
max_f1_diff = float(np.max(np.abs((f1_table - reference_f1).to_numpy())))
print("Reconstructed report Table 7, test F1 (%):")
print(f1_table.to_string(float_format=lambda x: f"{x:.2f}"))
print("Largest unrounded difference vs Table 7:", f"{max_f1_diff:.6f} pp")
assert max_f1_diff < 0.0051, "Per-genre F1 mismatch; stop before publishing figure"

FIG_DIR.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(2, 2, figsize=(13, 11), constrained_layout=True)
for ax, short in zip(axes.flat, FIGURE1_LABELS):
    cm = figure1_matrices[short][1]
    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=100)
    ax.set_title(short)
    ax.set_xticks(range(len(GENRES)), GENRES, rotation=45, ha="right")
    ax.set_yticks(range(len(GENRES)), GENRES)
    ax.set_xlabel("Predicted genre")
    ax.set_ylabel("True genre")
    for i in range(len(GENRES)):
        for j in range(len(GENRES)):
            ax.text(j, i, f"{cm[i, j]:.0f}", ha="center", va="center",
                    fontsize=9, color="white" if cm[i, j] > 50 else "black")
fig.colorbar(im, ax=axes, shrink=0.65, label="Percentage of true genre (%)")
fig.suptitle("Genre Classification — Row-Normalized Confusion Matrices")
fig.savefig(FIG_DIR / "stage5_figure1_confusion_matrices.png", dpi=250, bbox_inches="tight")
fig.savefig(FIG_DIR / "stage5_figure1_confusion_matrices.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)
print("Figure 1 generated independently and Table 7 verified.")

## 5. Reconstruct **Figure 2** (fusion accuracy gain by typicality)

Use the *verified* Stage 3 artist-cluster bootstrap intervals (3,000 resamples), rather than using a new random seed. Confirm all ten CNN/MM rows and their intervals against the report before plotting.

In [ ]:
quintile_ci = pd.read_csv(required["verified Stage 3 bootstrap intervals"])
ci_lookup = {
    "CNN fusion vs CNN audio": {
        "estimate": [17.45,12.32,10.64,-1.96,-19.72],
        "lower": [13.01,7.85,6.51,-5.92,-24.34],
        "upper": [22.12,16.85,14.78,2.06,-15.22],
    },
    "MM fusion vs MM audio": {
        "estimate": [17.45,12.89,8.40,-3.36,-16.94],
        "lower": [13.11,8.83,4.22,-7.04,-21.80],
        "upper": [21.88,17.08,12.50,0.27,-12.40],
    },
}
for name, expected in ci_lookup.items():
    subset = quintile_ci.loc[quintile_ci["comparison"].eq(name)].sort_values("quintile")
    assert subset["quintile"].tolist() == [1,2,3,4,5]
    for column, key in (("estimate_pp", "estimate"), ("CI_lower_pp", "lower"), ("CI_upper_pp", "upper")):
        assert np.max(np.abs(subset[column].to_numpy() - expected[key])) < 0.0051, (name, column)
    assert (subset["valid_bootstraps"].astype(int) == 3000).all()
print("Figure 2: all original bootstrap estimates and intervals VERIFIED")

fig, ax = plt.subplots(figsize=(9, 5.7))
for name, label, offset in [
    ("CNN fusion vs CNN audio", "CNN fusion − CNN audio", -0.07),
    ("MM fusion vs MM audio", "MM fusion − MM audio", 0.07),
]:
    s = quintile_ci.loc[quintile_ci.comparison.eq(name)].sort_values("quintile")
    x = s.quintile.to_numpy(dtype=float)
    y = s.estimate_pp.to_numpy(dtype=float)
    errs = np.vstack([y - s.CI_lower_pp.to_numpy(), s.CI_upper_pp.to_numpy() - y])
    ax.errorbar(x + offset, y, yerr=errs, fmt="o-", linewidth=2, capsize=4, markersize=6, label=label)
ax.axhline(0, color="gray", linestyle="--", linewidth=1)
ax.set_xticks([1,2,3,4,5], ["Q1\nMost typical", "Q2", "Q3", "Q4", "Q5\nMost atypical"])
ax.set_xlabel("Album-cover visual typicality (within true genre)")
ax.set_ylabel("Fusion − audio-only accuracy (percentage points)")
ax.set_ylim(-30, 28)
ax.set_xlim(0.7, 5.3)
ax.set_title("Fusion gain varies with album-cover visual typicality")
ax.grid(axis="y", alpha=0.2)
ax.legend(frameon=False)
fig.text(0.13, 0.015, "95% artist-cluster bootstrap intervals (3,000 resamples); N = 1,792.\nTypicality uses true genre: retrospective analysis.", fontsize=8.5)
fig.subplots_adjust(bottom=0.24, top=0.87)
fig.savefig(FIG_DIR / "stage5_figure2_fusion_gain_by_typicality.png", dpi=250, bbox_inches="tight")
fig.savefig(FIG_DIR / "stage5_figure2_fusion_gain_by_typicality.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)
print("Figure 2 saved separately; the original report's figure is unchanged.")

## 6. Reconstruct supplementary **Figure 3** — absolute accuracy by typicality

The report has **five curves**: CNN Audio LR, CNN Fusion LR, MM Fusion LR, CNN Fusion NN, and MM Fusion NN, all grouped using the **same CNN-derived true-genre typicality quintiles**.

In [ ]:
FIGURE3_MODELS = ["CNN Audio LR", "CNN Early Fusion LR", "MM Early Fusion LR", "CNN Fusion NN", "MM Fusion NN"]
fig3_rows = []
for model in FIGURE3_MODELS:
    for q in range(1,6):
        group = figure_data.loc[figure_data.typicality_quintile.eq(q)]
        acc = 100 * (group[model] == group["genre"]).mean()
        fig3_rows.append(dict(model=model, quintile=q, n=len(group), accuracy_pct=acc))
figure3_accuracy = pd.DataFrame(fig3_rows)
# Grounded cross-checks from the paper's supplementary analysis.
checks = {
    ("CNN Audio LR", 1): 71.75,
    ("CNN Audio LR", 5): 59.72,
    ("CNN Early Fusion LR", 1): 89.20,
    ("CNN Early Fusion LR", 5): 40.00,
    ("MM Early Fusion LR", 1): 90.30,
    ("MM Early Fusion LR", 5): 41.94,
}
for (model, q), expected in checks.items():
    current = figure3_accuracy.query("model == @model and quintile == @q").accuracy_pct.iloc[0]
    assert abs(current - expected) < 0.0051, (model, q, current, expected)
print("Figure 3: verified Q1/Q5 endpoint values against the original analysis")
print(figure3_accuracy.pivot(index="quintile", columns="model", values="accuracy_pct").round(2).to_string())
fig, ax = plt.subplots(figsize=(10, 5.7))
for model in FIGURE3_MODELS:
    s=figure3_accuracy.query("model == @model").sort_values("quintile")
    ax.plot(s.quintile, s.accuracy_pct, marker="o", linewidth=1.8, label=model.replace("Early ", ""))
ax.set_xticks([1,2,3,4,5], ["Q1", "Q2", "Q3", "Q4", "Q5"])
ax.set_xlabel("Visual typicality quintile (Q1 most typical, Q5 most atypical)")
ax.set_ylabel("Test accuracy (%)")
ax.set_title("Music Genre Classification Accuracy by Visual Typicality")
ax.grid(alpha=.2)
ax.legend(loc="upper right", fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / "stage5_figure3_accuracy_by_typicality.png", dpi=250, bbox_inches="tight")
fig.savefig(FIG_DIR / "stage5_figure3_accuracy_by_typicality.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

## 7. Verify supplementary **Tables 3, 4, 8, 9 and 10**

- Table 3: Q5 CNN late-fusion comparisons.
- Table 4: Q1–Q5 rescued/harmed counts and net gains.
- Table 8: Overall paired transition counts.
- Table 9: Audio–visual prediction disagreement and harmed predictions that match the visual-only classifier.
- Table 10: Exclusion sensitivity for candidate near-duplicate artwork.

The report values are fixed references, not targets for tuning.

In [ ]:
# Table 3, Q5 (original predictions).
q5 = figure_data.query("typicality_quintile == 5").copy()
q5_ref = {"CNN Audio LR": (59.72, None, None),
          "CNN Early Fusion LR": (40.00, 10, 81),
          "CNN Fixed Late Fusion": (46.67, 3, 50),
          "CNN Adaptive Late Fusion": (51.39, 1, 31)}
q5_rows = []
audio_correct = q5["CNN Audio LR"].eq(q5.genre)
for model, (ref_acc, ref_rescued, ref_harmed) in q5_ref.items():
    new_correct = q5[model].eq(q5.genre)
    acc = 100 * new_correct.mean()
    rescued = int((~audio_correct & new_correct).sum())
    harmed = int((audio_correct & ~new_correct).sum())
    assert abs(acc - ref_acc) < .0051, model
    if ref_rescued is not None:
        assert (rescued, harmed) == (ref_rescued, ref_harmed), model
    q5_rows.append(dict(model=model, n=len(q5), accuracy_pct=acc,
                        rescued=None if ref_rescued is None else rescued,
                        harmed=None if ref_harmed is None else harmed))
table3_q5 = pd.DataFrame(q5_rows)
print("Table 3 (Q5 strategies): VERIFIED")
print(table3_q5.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# Tables 4 and 8, using independent saved Stage 3 counts.
transitions = pd.read_csv(required["verified Stage 3 quintile transitions"])
overall_transitions = pd.read_csv(required["verified Stage 3 overall transitions"])
expected_overall = {"CNN": (1039,209,142,402), "MM": (1068,190,124,410)}
for rep, counts in expected_overall.items():
    row = overall_transitions.query("representation == @rep").iloc[0]
    assert tuple(int(row[k]) for k in ("both_correct","rescued","harmed","both_wrong")) == counts
    sub = transitions.query("representation == @rep").sort_values("quintile")
    assert sub.quintile.tolist() == [1,2,3,4,5]
    assert int(sub.rescued.sum()) == counts[1] and int(sub.harmed.sum()) == counts[2]
    assert np.allclose(100*(sub.rescued-sub.harmed)/sub.N, sub.net_gain_pp)
print("Tables 4 and 8: quintile and overall transitions VERIFIED")

# Table 9: visual-only disagreements and harmed matching visual prediction.
t9_rows=[]
for q in range(1,6):
    group = figure_data.query("typicality_quintile == @q")
    disagree = 100 * (group["CNN Audio LR"] != group["CNN Visual LR"]).mean()
    harmed = group[(group["CNN Audio LR"] == group.genre) & (group["CNN Early Fusion LR"] != group.genre)]
    agrees = int((harmed["CNN Early Fusion LR"] == harmed["CNN Visual LR"]).sum())
    t9_rows.append(dict(quintile=q, disagreement_pct=disagree,
                        harmed_matching_visual=agrees, harmed_total=len(harmed)))
table9 = pd.DataFrame(t9_rows)
reported_disagree = [33.24,49.02,56.02,71.99,78.61]
reported_match = [5,9,8,25,63]
reported_harmed = [8,10,11,32,81]
assert np.max(np.abs(table9.disagreement_pct.to_numpy()-reported_disagree)) < .0051
assert table9.harmed_matching_visual.tolist() == reported_match
assert table9.harmed_total.tolist() == reported_harmed
print("Table 9: visual prediction agreement VERIFIED")
print(table9.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# Table 10, sensitivity using preserved Stage 4 results.
sensitivity = pd.read_csv(required["verified Stage 4 overall sensitivity"])
sensitivity_q = pd.read_csv(required["verified Stage 4 quintile sensitivity"])
cnn_full = sensitivity.query("representation == 'CNN' and group == 'Full test set'").iloc[0]
cnn_clean = sensitivity.query("representation == 'CNN' and group == 'Excluding flagged covers'").iloc[0]
assert abs(cnn_full.fusion_gain_pp - 3.74) < .0051
assert abs(cnn_clean.fusion_gain_pp - 3.64) < .0051
for q, before, after in ((1,17.45,17.04), (5,-19.72,-19.72)):
    row=sensitivity_q.query("representation == 'CNN' and quintile == @q").iloc[0]
    assert abs(row.gain_full_pp - before) < .0051
    assert abs(row.gain_excluding_flagged_pp - after) < .0051
print("Table 10: original robustness values VERIFIED")

## 8. Audit **Figure 4** — selected real album covers (metadata only)

This section verifies the **six selected tracks and their prediction outcomes** against the original saved predictions. It **does not download/reuse imagery**. The six images in the paper still require a human visual comparison against the saved `figure3_album_cover_examples.pdf`; this notebook cannot prove the displayed image bytes match the metadata.

In [ ]:
example_path = ORIGINAL_EXAMPLES_DIR / "final_report_album_examples.csv"
if example_path.is_file():
    example = pd.read_csv(example_path, dtype={"msd_track_id": str})
    expected_genres = ["Pop", "Rap", "Jazz", "Rock", "Rap", "Jazz"]
    assert len(example) == 6 and example.msd_track_id.is_unique
    assert example.genre_typicality.tolist() == expected_genres
    assert example.example_group.tolist() == ["Q1_both_rescued"]*3 + ["Q5_both_harmed"]*3
    example_checks = example.merge(
        figure_data[["msd_track_id", "genre", "typicality_quintile",
                     "CNN Audio LR", "CNN Early Fusion LR", "MM Audio LR", "MM Early Fusion LR"]],
        on="msd_track_id", how="left", validate="one_to_one", sort=False)
    assert example_checks["genre"].notna().all()
    assert (example_checks.genre_typicality == example_checks.genre).all()
    for src, model in (("cnn_audio_prediction", "CNN Audio LR"),
                       ("cnn_early_fusion_prediction", "CNN Early Fusion LR"),
                       ("mm_audio_prediction", "MM Audio LR"),
                       ("mm_fusion_prediction", "MM Early Fusion LR")):
        assert (example_checks[src] == example_checks[model]).all(), src
    for row in example_checks.itertuples(index=False):
        rescued = row.example_group == "Q1_both_rescued"
        assert row.typicality_quintile == (1 if rescued else 5)
        cnn_audio_correct = getattr(row, 'cnn_audio_prediction') == row.genre
        cnn_fusion_correct = getattr(row, 'cnn_early_fusion_prediction') == row.genre
        mm_audio_correct = getattr(row, 'mm_audio_prediction') == row.genre
        mm_fusion_correct = getattr(row, 'mm_fusion_prediction') == row.genre
        assert (cnn_audio_correct, cnn_fusion_correct, mm_audio_correct, mm_fusion_correct) == (
            (False,True,False,True) if rescued else (True,False,True,False))
    print("Figure 4 selection: 6 track IDs / genres / predictions VERIFIED")
    print("Image file must still be visually reviewed:", ORIGINAL_FIG_DIR / "figure3_album_cover_examples.pdf")
    print("No image URLs were fetched or redistributed.")
else:
    print("Figure 4 original six-example CSV unavailable; needs manual check:", example_path)

## 9. Save new figures and a concise audit trail

Only save **after** the preceding checks have passed. Export a figure-source CSV and comparison tables. These outputs reflect the original frozen predictions, not the slightly different Stage 1 retraining results.

### Report mapping

| Report | Reconstructed source |
|---|---|
| Table 2 | `stage5_table2_model_metrics.csv` |
| Figure 1 | `figures/stage5_figure1_confusion_matrices.pdf` |
| Table 7 | `stage5_table7_per_genre_f1.csv` |
| Figure 2 | `figures/stage5_figure2_fusion_gain_by_typicality.pdf` |
| Figure 3 (Appendix C) | `figures/stage5_figure3_accuracy_by_typicality.pdf` |
| Table 3 | `stage5_table3_q5_fusion_strategies.csv` |
| Table 9 | `stage5_table9_visual_prediction_agreement.csv` |
| Figure 4 (Appendix D) | Six-example source metadata checked separately; visual review still required |

Review the displayed figures side-by-side with the report PDFs before calling their visual formatting verified. A match in underlying numbers is not automatically a pixel-level match.

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
metrics.to_csv(OUTPUT / "stage5_table2_model_metrics.csv", index=False)
f1_table.to_csv(OUTPUT / "stage5_table7_per_genre_f1.csv")
f1_detail.to_csv(OUTPUT / "stage5_table7_per_genre_f1_detail.csv", index=False)
figure3_accuracy.to_csv(OUTPUT / "stage5_figure3_accuracy_source.csv", index=False)
quintile_ci.to_csv(OUTPUT / "stage5_figure2_bootstrap_source.csv", index=False)
table3_q5.to_csv(OUTPUT / "stage5_table3_q5_fusion_strategies.csv", index=False)
table9.to_csv(OUTPUT / "stage5_table9_visual_prediction_agreement.csv", index=False)
# Export all four row-normalized confusion matrices in long format.
cm_rows=[]
for short, (counts, percents) in figure1_matrices.items():
    for i,true in enumerate(GENRES):
        for j,pred in enumerate(GENRES):
            cm_rows.append(dict(model=short,true_genre=true,predicted_genre=pred,
                                count=int(counts[i,j]),percentage=float(percents[i,j])))
pd.DataFrame(cm_rows).to_csv(OUTPUT / "stage5_figure1_confusion_matrix_source.csv", index=False)

print("Stage 5 numeric audit saved to:", OUTPUT)
print("Three newly generated figures (PDF and PNG) saved to:", FIG_DIR)
print("All required numeric tables match the report at its stated precision.")
print("IMPORTANT: Figure 4 artwork and figure layout still need human visual inspection.")
print("No existing research or prior-stage files were overwritten.")

## Verification boundaries

- **Numeric consistency:** This notebook checks original prediction files, report tables, confusion matrices, and reproduced Stage 3/4 uncertainty and robustness outputs. It does not substitute new model training for the archived original test predictions.
- **Statistical uncertainty:** It reuses the already reproduced Stage 3 3,000-artist-bootstrap estimates; it does **not** claim a second independently generated bootstrap run in Stage 5.
- **Image provenance:** Figure 4 is a metadata/prediction audit only; review image content/availability separately.
- **Scientific limitation:** Visual typicality uses true test genre, and the analysis is retrospective and exploratory. It is not a deployable classifier confidence score or evidence of causal image effects.
- **Paper version:** References reflect the 21-page November 2026 report used to construct this notebook. If the team revises the paper, redo the audit against that revision.

**Next:** record the results under `REPRODUCIBILITY_VERIFICATION.md`, update the GitHub README with how to run Stages 1–5, and perform a clean-environment reproduction check. Do not commit Google Drive credentials, private folders, or large source archives.